# 🧵 Textile Defect Detection — Dataset Preprocessing Pipeline
## Hangzhou 2026 POC | CHENAB Textile / FabricDefectNTU

---

## Section 1 — Project Overview

### Project: Textile Defect Detection — Hangzhou 2026 Proof of Concept

This notebook is the **main preprocessing record** for the Hangzhou 2026 Textile Defect Detection Proof of Concept (POC).  
It documents every step of dataset preparation from raw download to a training-ready YOLOv8 dataset.

| Item | Detail |
|---|---|
| **Project** | Textile Defect Detection — Hangzhou 2026 POC |
| **Dataset** | CHENAB Textile / FabricDefectNTU |
| **Kaggle ID** | `muhammadharisabid/fabricdefectntu` |
| **Source** | [Kaggle Dataset](https://www.kaggle.com/datasets/muhammadharisabid/fabricdefectntu) |
| **Target Framework** | YOLOv8 (Ultralytics) |
| **Expected Output** | `final_dataset/` + `textile_defect_yolov8_final.zip` |
| **Organization** | Devlogix Technology |

### Why this dataset?

The FabricDefectNTU dataset provides real-world textile/fabric defect images with pre-labeled YOLO-format annotations  
covering multiple defect categories. It is open-source, well-structured, and directly compatible with YOLOv8 training.

### Expected final output

```
final_dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── data.yaml
```

---
## Section 2 — Environment Setup

In [ ]:
# Install required packages
import subprocess, sys
packages = [
    'kagglehub', 'ultralytics', 'opencv-python', 'Pillow', 'numpy', 'pandas',
    'matplotlib', 'albumentations', 'pyyaml', 'tqdm', 'imagehash', 'nbformat'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages, check=False)
print('Package installation attempted.')

In [ ]:
# Core imports
import os
import sys
import shutil
import hashlib
import json
import csv
import random
import math
import warnings
import datetime
from pathlib import Path
from collections import Counter, defaultdict

# Data science
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Computer vision
import cv2
from PIL import Image
import imagehash

# ML / dataset
import yaml
from tqdm import tqdm
import albumentations as A

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100
random.seed(42)
np.random.seed(42)

# ── Project root ──────────────────────────────────────────────────────────────
# Notebook lives in notebooks/, project root is one level up
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Reports dir  : {REPORTS_DIR}')
print(f'Python       : {sys.version}')

import ultralytics
print(f'Ultralytics  : {ultralytics.__version__}')
print(f'OpenCV       : {cv2.__version__}')
print(f'NumPy        : {np.__version__}')
print(f'Albumentations: {A.__version__}')

---
## Section 3 — Dataset Download

In [ ]:
import kagglehub

DOWNLOAD_DATE = datetime.datetime.now().isoformat()
KAGGLE_DATASET_ID = 'muhammadharisabid/fabricdefectntu'

print(f'Kaggle dataset : {KAGGLE_DATASET_ID}')
print(f'Download date  : {DOWNLOAD_DATE}')
print('Downloading...')

RAW_DATASET_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
RAW_DATASET_PATH = Path(RAW_DATASET_PATH)

print(f'Path to dataset files: {RAW_DATASET_PATH}')

# Save download metadata
download_meta = {
    'kaggle_dataset_id': KAGGLE_DATASET_ID,
    'download_date': DOWNLOAD_DATE,
    'raw_path': str(RAW_DATASET_PATH),
    'kaggle_url': f'https://www.kaggle.com/datasets/{KAGGLE_DATASET_ID}'
}
with open(REPORTS_DIR / 'download_metadata.json', 'w') as f:
    json.dump(download_meta, f, indent=2)
print('Metadata saved to reports/download_metadata.json')

---
## Section 4 — Dataset Structure Inspection

In [ ]:
def print_tree(path: Path, prefix: str = '', max_files_per_dir: int = 5):
    """Print directory tree with truncation."""
    items = sorted(path.iterdir())
    dirs = [i for i in items if i.is_dir()]
    files = [i for i in items if i.is_file()]
    shown = dirs + files[:max_files_per_dir]
    hidden = max(0, len(files) - max_files_per_dir)
    for i, item in enumerate(shown):
        connector = '└── ' if (i == len(shown) - 1 and hidden == 0) else '├── '
        print(f'{prefix}{connector}{item.name}' + ('/' if item.is_dir() else ''))
        if item.is_dir():
            extension = '    ' if (i == len(shown) - 1 and hidden == 0) else '│   '
            print_tree(item, prefix + extension, max_files_per_dir)
    if hidden > 0:
        print(f'{prefix}└── ... and {hidden} more files')

print(f'\n📂 Dataset root: {RAW_DATASET_PATH}')
print('='*60)
print_tree(RAW_DATASET_PATH)
print('='*60)

In [ ]:
# ── Locate data.yaml ─────────────────────────────────────────────────────────
yaml_candidates = list(RAW_DATASET_PATH.rglob('data.yaml'))
yaml_candidates += list(RAW_DATASET_PATH.rglob('*.yaml'))
yaml_candidates = sorted(set(yaml_candidates))

print('YAML files found:')
for yc in yaml_candidates:
    print(f'  {yc}')

# Use data.yaml if present, else first yaml
DATA_YAML_RAW = None
for yc in yaml_candidates:
    if yc.name == 'data.yaml':
        DATA_YAML_RAW = yc
        break
if DATA_YAML_RAW is None and yaml_candidates:
    DATA_YAML_RAW = yaml_candidates[0]

if DATA_YAML_RAW:
    print(f'\nUsing YAML: {DATA_YAML_RAW}')
    with open(DATA_YAML_RAW, 'r') as f:
        raw_cfg = yaml.safe_load(f)
    print('\ndata.yaml contents:')
    print(json.dumps(raw_cfg, indent=2, default=str))
    RAW_CLASS_NAMES = raw_cfg.get('names', [])
    NUM_CLASSES = len(RAW_CLASS_NAMES)
    print(f'\nClasses ({NUM_CLASSES}): {RAW_CLASS_NAMES}')
else:
    print('[WARNING] No data.yaml found — will infer classes from annotations')
    RAW_CLASS_NAMES = []
    NUM_CLASSES = 0
    raw_cfg = {}

In [ ]:
# ── Discover splits ───────────────────────────────────────────────────────────
SPLIT_NAMES = ['train', 'valid', 'val', 'test']
SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def find_split_dirs(root: Path):
    """Auto-discover image and label directories for each split."""
    splits = {}
    # Check common structures
    for split in SPLIT_NAMES:
        img_candidates = [
            root / 'images' / split,
            root / split / 'images',
            root / split,
        ]
        lbl_candidates = [
            root / 'labels' / split,
            root / split / 'labels',
        ]
        img_dir = next((p for p in img_candidates if p.exists() and any(
            f.suffix.lower() in SUPPORTED_EXTS for f in p.iterdir() if f.is_file())), None)
        lbl_dir = next((p for p in lbl_candidates if p.exists()), None)
        if img_dir:
            splits[split] = {'images': img_dir, 'labels': lbl_dir}
    return splits

RAW_SPLITS = find_split_dirs(RAW_DATASET_PATH)

# If dataset has nested dirs, search one level deeper
if not RAW_SPLITS:
    for subdir in RAW_DATASET_PATH.iterdir():
        if subdir.is_dir():
            RAW_SPLITS = find_split_dirs(subdir)
            if RAW_SPLITS:
                RAW_DATASET_PATH = subdir
                print(f'Adjusted dataset root to: {RAW_DATASET_PATH}')
                break

print('\nDiscovered splits:')
for split, dirs in RAW_SPLITS.items():
    img_count = sum(1 for f in dirs['images'].iterdir() if f.suffix.lower() in SUPPORTED_EXTS) if dirs['images'] else 0
    lbl_dir = dirs.get('labels')
    lbl_count = sum(1 for f in lbl_dir.iterdir() if f.suffix == '.txt') if lbl_dir and lbl_dir.exists() else 0
    print(f'  {split:8s}: {img_count:4d} images | {lbl_count:4d} labels')
    print(f'             images → {dirs["images"]}')
    print(f'             labels → {lbl_dir}')

---
## Section 5 — Dataset Statistics

In [ ]:
# ── Full statistics collection ────────────────────────────────────────────────
stats = {
    'splits': {},
    'resolutions': [],
    'formats': Counter(),
    'class_counts': Counter(),
    'total_images': 0,
    'total_annotations': 0,
    'corrupted': [],
    'missing_labels': [],
    'empty_labels': [],
}

for split, dirs in RAW_SPLITS.items():
    img_dir = dirs['images']
    lbl_dir = dirs.get('labels')
    image_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
    
    split_stats = {
        'images': [], 'annotations': 0, 'class_counts': Counter(),
        'missing_labels': [], 'empty_labels': [], 'corrupted': []
    }
    
    for img_path in tqdm(image_files, desc=f'Scanning {split}'):
        img = cv2.imread(str(img_path))
        if img is None:
            stats['corrupted'].append(str(img_path))
            split_stats['corrupted'].append(img_path.name)
            continue
        h, w = img.shape[:2]
        stats['resolutions'].append((w, h))
        stats['formats'][img_path.suffix.lower()] += 1
        split_stats['images'].append({'name': img_path.name, 'w': w, 'h': h, 'path': str(img_path)})
        
        if lbl_dir and lbl_dir.exists():
            lbl = lbl_dir / (img_path.stem + '.txt')
            if not lbl.exists():
                stats['missing_labels'].append(img_path.name)
                split_stats['missing_labels'].append(img_path.name)
            else:
                lines = [l.strip() for l in lbl.read_text(encoding='utf-8').splitlines() if l.strip()]
                if not lines:
                    stats['empty_labels'].append(img_path.name)
                    split_stats['empty_labels'].append(img_path.name)
                else:
                    for line in lines:
                        try:
                            cls_id = int(line.split()[0])
                            stats['class_counts'][cls_id] += 1
                            split_stats['class_counts'][cls_id] += 1
                            split_stats['annotations'] += 1
                            stats['total_annotations'] += 1
                        except (ValueError, IndexError):
                            pass
    
    stats['splits'][split] = split_stats
    stats['total_images'] += len(split_stats['images'])

# ── Print summary table ───────────────────────────────────────────────────────
print('\n' + '='*60)
print('DATASET STATISTICS')
print('='*60)
print(f'Total images      : {stats["total_images"]}')
print(f'Total annotations : {stats["total_annotations"]}')
print(f'Classes           : {NUM_CLASSES} → {RAW_CLASS_NAMES}')
print(f'Image formats     : {dict(stats["formats"])}')
print(f'Corrupted images  : {len(stats["corrupted"])}')
print(f'Missing labels    : {len(stats["missing_labels"])}')
print(f'Empty labels      : {len(stats["empty_labels"])}')
print()
print(f'{"Split":<10} {"Images":>8} {"Annotations":>14}')
print('-'*36)
for split, sd in stats['splits'].items():
    print(f'{split:<10} {len(sd["images"]):>8} {sd["annotations"]:>14}')
print()
print('Class distribution:')
for cls_id, cnt in sorted(stats['class_counts'].items()):
    name = RAW_CLASS_NAMES[cls_id] if cls_id < len(RAW_CLASS_NAMES) else f'cls_{cls_id}'
    print(f'  [{cls_id}] {name:<25}: {cnt:>5}')
print('='*60)

In [ ]:
# ── Resolution statistics ─────────────────────────────────────────────────────
if stats['resolutions']:
    widths = [r[0] for r in stats['resolutions']]
    heights = [r[1] for r in stats['resolutions']]
    print('Resolution Statistics:')
    print(f'  Width  — min:{min(widths):5d}  max:{max(widths):5d}  median:{int(np.median(widths)):5d}  mean:{np.mean(widths):.1f}')
    print(f'  Height — min:{min(heights):5d}  max:{max(heights):5d}  median:{int(np.median(heights)):5d}  mean:{np.mean(heights):.1f}')
    print(f'  Unique sizes: {len(set(stats["resolutions"]))}')
    
    # Plot resolution distribution
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title('Image Width Distribution')
    axes[0].set_xlabel('Width (px)')
    axes[0].set_ylabel('Count')
    
    axes[1].hist(heights, bins=30, color='darkorange', edgecolor='white')
    axes[1].set_title('Image Height Distribution')
    axes[1].set_xlabel('Height (px)')
    axes[1].set_ylabel('Count')
    
    # Class distribution bar chart
    cls_ids = sorted(stats['class_counts'].keys())
    cls_names_short = [RAW_CLASS_NAMES[c][:15] if c < len(RAW_CLASS_NAMES) else f'cls_{c}' for c in cls_ids]
    cls_vals = [stats['class_counts'][c] for c in cls_ids]
    axes[2].bar(cls_names_short, cls_vals, color='mediumseagreen', edgecolor='white')
    axes[2].set_title('Annotations per Class')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('Count')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(str(REPORTS_DIR / 'dataset_statistics.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('Plot saved to reports/dataset_statistics.png')

---
## Section 6 — Visual Dataset Inspection

In [ ]:
def draw_yolo_boxes(img_bgr, label_path, class_names, color_map=None):
    """Draw YOLO bounding boxes on a copy of img_bgr."""
    img = img_bgr.copy()
    h, w = img.shape[:2]
    if not label_path.exists():
        return img
    lines = [l.strip() for l in label_path.read_text(encoding='utf-8').splitlines() if l.strip()]
    palette = plt.get_cmap('tab10')
    for line in lines:
        parts = line.split()
        if len(parts) < 5:
            continue
        cls_id = int(float(parts[0]))
        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        color_rgb = palette(cls_id % 10)[:3]
        color_bgr = (int(color_rgb[2]*255), int(color_rgb[1]*255), int(color_rgb[0]*255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color_bgr, 2)
        label = class_names[cls_id] if cls_id < len(class_names) else f'cls_{cls_id}'
        cv2.putText(img, label, (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_bgr, 1, cv2.LINE_AA)
    return img

def show_samples(split_name, img_dir, lbl_dir, class_names, n=6):
    image_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
    samples = random.sample(image_files, min(n, len(image_files)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Visual Inspection — Split: {split_name}', fontsize=14, fontweight='bold')
    for ax, img_path in zip(axes.flat, samples):
        img = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, 'CORRUPTED', ha='center', va='center', color='red')
            ax.set_title(img_path.name[:30])
            continue
        if lbl_dir and lbl_dir.exists():
            lbl = lbl_dir / (img_path.stem + '.txt')
            img = draw_yolo_boxes(img, lbl, class_names)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(f'{img_path.name[:25]} ({img.shape[1]}×{img.shape[0]})', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(str(REPORTS_DIR / f'visual_inspection_{split_name}.png'), dpi=80, bbox_inches='tight')
    plt.show()
    print(f'Saved to reports/visual_inspection_{split_name}.png')

for split, dirs in RAW_SPLITS.items():
    show_samples(split, dirs['images'], dirs.get('labels'), RAW_CLASS_NAMES)

---
## Section 7 — Annotation Validation

In [ ]:
def validate_annotation_file(lbl_path: Path, num_classes: int) -> list:
    """Returns list of issues found. Empty list = valid."""
    issues = []
    try:
        text = lbl_path.read_text(encoding='utf-8')
    except Exception as e:
        return [f'Cannot read: {e}']
    
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for line_no, line in enumerate(lines, 1):
        parts = line.split()
        if len(parts) != 5:
            issues.append(f'L{line_no}: expected 5 fields, got {len(parts)}')
            continue
        try:
            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        except ValueError:
            issues.append(f'L{line_no}: non-numeric values')
            continue
        for val in [xc, yc, bw, bh]:
            if math.isnan(val) or math.isinf(val):
                issues.append(f'L{line_no}: NaN/Inf in coordinates')
        if cls_id < 0 or cls_id >= num_classes:
            issues.append(f'L{line_no}: invalid class_id={cls_id} (valid 0-{num_classes-1})')
        if not (0 <= xc <= 1 and 0 <= yc <= 1):
            issues.append(f'L{line_no}: x_center/y_center out of [0,1]: {xc:.4f},{yc:.4f}')
        if not (0 < bw <= 1 and 0 < bh <= 1):
            issues.append(f'L{line_no}: width/height out of (0,1]: {bw:.4f},{bh:.4f}')
    return issues

# Run validation on all splits
validation_results = []
total_valid = 0
total_invalid = 0

for split, dirs in RAW_SPLITS.items():
    lbl_dir = dirs.get('labels')
    img_dir = dirs['images']
    if not lbl_dir or not lbl_dir.exists():
        print(f'[{split}] No label directory')
        continue
    
    image_stems = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS}
    label_files = sorted(f for f in lbl_dir.iterdir() if f.suffix == '.txt')
    
    split_valid = 0
    split_invalid = 0
    for lbl_path in tqdm(label_files, desc=f'Validating {split} labels'):
        issues = validate_annotation_file(lbl_path, NUM_CLASSES)
        has_image = lbl_path.stem in image_stems
        entry = {
            'split': split,
            'filename': lbl_path.name,
            'has_image': has_image,
            'issues': '; '.join(issues) if issues else 'VALID',
            'valid': len(issues) == 0
        }
        validation_results.append(entry)
        if issues:
            split_invalid += 1
        else:
            split_valid += 1
    
    total_valid += split_valid
    total_invalid += split_invalid
    print(f'[{split}] Valid: {split_valid} | Invalid: {split_invalid}')

print(f'\nOVERALL — Valid: {total_valid} | Invalid: {total_invalid}')

# Save validation report
df_val = pd.DataFrame(validation_results)
df_val.to_csv(str(REPORTS_DIR / 'annotation_validation.csv'), index=False)
print('Saved to reports/annotation_validation.csv')

# Show invalid entries
invalid_df = df_val[~df_val['valid']]
if len(invalid_df) > 0:
    print('\nInvalid annotations:')
    print(invalid_df[['split','filename','issues']].to_string())
else:
    print('\n✅ All annotation files passed validation!')

---
## Section 8 — Image Cleaning

In [ ]:
cleaning_log = []
EXCLUDED_IMAGES = set()  # stems to exclude from final dataset

for split, dirs in RAW_SPLITS.items():
    img_dir = dirs['images']
    lbl_dir = dirs.get('labels')
    image_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
    
    for img_path in tqdm(image_files, desc=f'Cleaning {split}'):
        issues = []
        action = 'KEEP'
        reason = ''
        
        # 1. Readable?
        img = cv2.imread(str(img_path))
        if img is None:
            issues.append('corrupted/unreadable')
            action = 'EXCLUDE'
            reason = 'Image cannot be decoded by OpenCV'
            EXCLUDED_IMAGES.add(img_path.stem)
        else:
            h, w = img.shape[:2]
            # 2. Zero-size?
            if h == 0 or w == 0:
                issues.append('zero-size')
                action = 'EXCLUDE'
                reason = 'Zero-dimension image'
                EXCLUDED_IMAGES.add(img_path.stem)
            # 3. Unsupported format?
            if img_path.suffix.lower() not in SUPPORTED_EXTS:
                issues.append('unsupported format')
        
        # 4. Missing label?
        if lbl_dir and lbl_dir.exists():
            lbl = lbl_dir / (img_path.stem + '.txt')
            if not lbl.exists():
                issues.append('missing_label')
                # Don't exclude — background images with no annotation are valid
                reason = reason or 'No label file — treated as background image'
        
        cleaning_log.append({
            'split': split,
            'filename': img_path.name,
            'issue_type': ', '.join(issues) if issues else 'none',
            'action_taken': action,
            'reason': reason or 'No issues found'
        })

# Save cleaning log
df_clean = pd.DataFrame(cleaning_log)
df_clean.to_csv(str(REPORTS_DIR / 'cleaning_log.csv'), index=False)

excluded = df_clean[df_clean['action_taken'] == 'EXCLUDE']
print(f'\nCleaning complete:')
print(f'  Total files checked : {len(df_clean)}')
print(f'  Excluded            : {len(excluded)}')
print(f'  Kept                : {len(df_clean) - len(excluded)}')
if len(excluded) > 0:
    print('\nExcluded files:')
    print(excluded[['split','filename','issue_type','reason']].to_string())
else:
    print('✅ No files need to be excluded!')
print('Saved to reports/cleaning_log.csv')

---
## Section 9 — Duplicate Detection

In [ ]:
def compute_md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

def compute_phash(path: Path) -> str:
    try:
        img = Image.open(path).convert('L')
        return str(imagehash.phash(img))
    except Exception:
        return ''

# Collect hashes per split
split_hashes = {}  # split → {md5: path}
duplicate_report = []

print('Computing image hashes...')
for split, dirs in RAW_SPLITS.items():
    img_dir = dirs['images']
    image_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
    hashes = {}
    within_dupes = 0
    
    for img_path in tqdm(image_files, desc=f'Hashing {split}'):
        md5 = compute_md5(img_path)
        if md5 in hashes:
            within_dupes += 1
            duplicate_report.append({
                'type': f'WITHIN_{split.upper()}',
                'file1': hashes[md5],
                'file2': str(img_path),
                'action': 'LOG_ONLY'
            })
        else:
            hashes[md5] = str(img_path)
    
    split_hashes[split] = hashes
    print(f'  [{split}] {len(image_files)} images, {within_dupes} within-split duplicates')

# Cross-split check
print('\nChecking cross-split duplicates (data leakage check)...')
cross_dupes = 0
split_list = list(split_hashes.keys())
for i in range(len(split_list)):
    for j in range(i + 1, len(split_list)):
        s1, s2 = split_list[i], split_list[j]
        common = set(split_hashes[s1].keys()) & set(split_hashes[s2].keys())
        for h in common:
            cross_dupes += 1
            duplicate_report.append({
                'type': f'CROSS_{s1.upper()}_{s2.upper()}',
                'file1': split_hashes[s1][h],
                'file2': split_hashes[s2][h],
                'action': 'EXCLUDE_FROM_FINAL'
            })
            EXCLUDED_IMAGES.add(Path(split_hashes[s2][h]).stem)  # exclude from second split
            print(f'  ⚠️  LEAKAGE: {Path(split_hashes[s1][h]).name} appears in {s1} AND {s2}')

if cross_dupes == 0:
    print('  ✅ No cross-split duplicates found — no data leakage!')

# Save report
df_dupes = pd.DataFrame(duplicate_report)
if not df_dupes.empty:
    df_dupes.to_csv(str(REPORTS_DIR / 'duplicate_report.csv'), index=False)
print(f'\nTotal duplicates found: {len(duplicate_report)}')
print(f'Cross-split (leakage): {cross_dupes}')

---
## Section 10 — Image Resolution Analysis

In [ ]:
if stats['resolutions']:
    widths = [r[0] for r in stats['resolutions']]
    heights = [r[1] for r in stats['resolutions']]
    unique_res = Counter(stats['resolutions'])
    
    print('Resolution Analysis:')
    print(f'  Unique resolutions : {len(unique_res)}')
    print(f'  Width  range       : {min(widths)} – {max(widths)} px (median {int(np.median(widths))})')
    print(f'  Height range       : {min(heights)} – {max(heights)} px (median {int(np.median(heights))})')
    print(f'\n  Top 10 resolutions:')
    for res, cnt in sorted(unique_res.items(), key=lambda x: -x[1])[:10]:
        print(f'    {res[0]}×{res[1]}: {cnt} images')
    
    # Decision on normalization
    dominant_w, dominant_h = max(unique_res, key=unique_res.get)
    pct_dominant = unique_res[(dominant_w, dominant_h)] / len(stats['resolutions']) * 100
    
    print(f'\nNormalization Decision:')
    if pct_dominant > 90:
        NORMALIZE_IMAGES = False
        print(f'  ✅ {pct_dominant:.1f}% of images are already {dominant_w}×{dominant_h}')
        print(f'  DECISION: Normalization NOT needed.')
        print(f'  YOLOv8 will handle resizing internally (letterboxing).')
    elif len(unique_res) <= 3:
        NORMALIZE_IMAGES = False
        print(f'  Only {len(unique_res)} unique resolution(s) — very consistent.')
        print(f'  DECISION: Normalization NOT needed.')
    else:
        NORMALIZE_IMAGES = False  # Still defer to YOLOv8 unless extreme
        print(f'  Multiple resolutions present, but YOLOv8 handles letterboxing internally.')
        print(f'  DECISION: Normalization NOT performed — YOLOv8 is resolution-agnostic.')
    
    print(f'\n  NORMALIZE_IMAGES = {NORMALIZE_IMAGES}')

---
## Section 11 — Create Clean Dataset

In [ ]:
# ── Setup final_dataset directory ─────────────────────────────────────────────
FINAL_DATASET = PROJECT_ROOT / 'final_dataset'

# Normalize 'valid' → 'val'
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

for target_split in ['train', 'val', 'test']:
    (FINAL_DATASET / 'images' / target_split).mkdir(parents=True, exist_ok=True)
    (FINAL_DATASET / 'labels' / target_split).mkdir(parents=True, exist_ok=True)

copy_counts = defaultdict(lambda: {'images': 0, 'labels': 0, 'skipped': 0})

for split, dirs in RAW_SPLITS.items():
    target_split = SPLIT_MAP.get(split, split)
    img_dir = dirs['images']
    lbl_dir = dirs.get('labels')
    img_out = FINAL_DATASET / 'images' / target_split
    lbl_out = FINAL_DATASET / 'labels' / target_split
    
    image_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
    
    for img_path in tqdm(image_files, desc=f'Copying {split} → {target_split}'):
        # Skip excluded images
        if img_path.stem in EXCLUDED_IMAGES:
            copy_counts[target_split]['skipped'] += 1
            continue
        
        dest_img = img_out / img_path.name
        if not dest_img.exists():
            shutil.copy2(img_path, dest_img)
        copy_counts[target_split]['images'] += 1
        
        if lbl_dir and lbl_dir.exists():
            lbl_src = lbl_dir / (img_path.stem + '.txt')
            if lbl_src.exists():
                dest_lbl = lbl_out / lbl_src.name
                if not dest_lbl.exists():
                    shutil.copy2(lbl_src, dest_lbl)
                copy_counts[target_split]['labels'] += 1

print('\nClean dataset created:')
for split, counts in copy_counts.items():
    print(f'  {split:8s}: {counts["images"]:4d} images | {counts["labels"]:4d} labels | {counts["skipped"]:4d} skipped')
print(f'  Location: {FINAL_DATASET}')

---
## Section 12 — Augmentation (Train Only)

In [ ]:
# ── Augmentation pipeline ─────────────────────────────────────────────────────
# Augmentation rationale for textile defect inspection:
#   - HorizontalFlip: valid (fabric can be inspected from either side)
#   - VerticalFlip: NOT used (defect orientation/gravity matters)
#   - BrightnessContrast: valid (lighting variation in factory)
#   - SmallRotation (±10°): valid (slight camera/fabric misalignment)
#   - MildScale: valid (camera distance variation)
#   - GaussNoise: valid (sensor noise)

AUG_TRANSFORM = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
        A.Rotate(limit=10, p=0.4),
        A.GaussNoise(std_range=(0.01, 0.04), p=0.3),
        A.RandomScale(scale_limit=0.1, p=0.3),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3,
        clip=True,
    )
)

AUG_COPIES_PER_IMAGE = 1  # 1 augmented copy per training image = 2× dataset
AUG_SUFFIX = '_aug'

print('Augmentation configuration:')
print(f'  HorizontalFlip          : p=0.5')
print(f'  RandomBrightnessContrast: brightness_limit=±0.2, contrast_limit=±0.2, p=0.7')
print(f'  Rotate                  : limit=±10°, p=0.4')
print(f'  GaussNoise              : std_range=(0.01,0.04), p=0.3')
print(f'  RandomScale             : scale_limit=±0.1, p=0.3')
print(f'  Copies per image        : {AUG_COPIES_PER_IMAGE}')
print(f'  Applied to              : train ONLY')
print(f'  Bounding boxes          : transformed together with images (YOLO format)')

In [ ]:
def read_yolo_labels(lbl_path: Path) -> tuple[list, list]:
    """Read YOLO label file. Returns (class_labels, bboxes)."""
    class_labels, bboxes = [], []
    if not lbl_path.exists():
        return class_labels, bboxes
    for line in lbl_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            try:
                cls_id = int(float(parts[0]))
                xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                # Clamp to valid range
                xc = max(0.001, min(0.999, xc))
                yc = max(0.001, min(0.999, yc))
                bw = max(0.001, min(1.0, bw))
                bh = max(0.001, min(1.0, bh))
                class_labels.append(cls_id)
                bboxes.append([xc, yc, bw, bh])
            except (ValueError, IndexError):
                continue
    return class_labels, bboxes

def write_yolo_labels(lbl_path: Path, class_labels: list, bboxes: list):
    """Write YOLO label file from class_labels and bboxes."""
    lines = []
    for cls_id, (xc, yc, bw, bh) in zip(class_labels, bboxes):
        if bw > 0 and bh > 0:
            lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    lbl_path.write_text('\n'.join(lines) + '\n' if lines else '', encoding='utf-8')

# Apply augmentation to train split
train_img_dir = FINAL_DATASET / 'images' / 'train'
train_lbl_dir = FINAL_DATASET / 'labels' / 'train'

train_images = sorted(f for f in train_img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
aug_success = 0
aug_skipped = 0

random.seed(42)
np.random.seed(42)

for img_path in tqdm(train_images, desc='Augmenting train'):
    img = cv2.imread(str(img_path))
    if img is None:
        aug_skipped += 1
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    class_labels, bboxes = read_yolo_labels(lbl_path)
    
    for copy_idx in range(AUG_COPIES_PER_IMAGE):
        try:
            if bboxes:
                result = AUG_TRANSFORM(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
                aug_img = result['image']
                aug_bboxes = result['bboxes']
                aug_labels = result['class_labels']
            else:
                # No bboxes — augment image only
                result = AUG_TRANSFORM(image=img_rgb, bboxes=[], class_labels=[])
                aug_img = result['image']
                aug_bboxes = []
                aug_labels = []
            
            aug_stem = f'{img_path.stem}{AUG_SUFFIX}{copy_idx}'
            aug_img_path = train_img_dir / f'{aug_stem}{img_path.suffix}'
            aug_lbl_path = train_lbl_dir / f'{aug_stem}.txt'
            
            cv2.imwrite(str(aug_img_path), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            write_yolo_labels(aug_lbl_path, aug_labels, list(aug_bboxes))
            aug_success += 1
        except Exception as e:
            aug_skipped += 1

print(f'\nAugmentation complete:')
print(f'  Original train images  : {len(train_images)}')
print(f'  Augmented copies added : {aug_success}')
print(f'  Failed/skipped         : {aug_skipped}')

# Count final train images
final_train_count = sum(1 for f in train_img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
print(f'  Final train total      : {final_train_count}')

---
## Section 13 — Augmentation Visualization

In [ ]:
# Show 3 original + augmented pairs
orig_images = [f for f in sorted(train_img_dir.iterdir())
               if f.suffix.lower() in SUPPORTED_EXTS and AUG_SUFFIX not in f.stem]

aug_image_map = {}
for f in train_img_dir.iterdir():
    if AUG_SUFFIX in f.stem and f.suffix.lower() in SUPPORTED_EXTS:
        orig_stem = f.stem.rsplit(AUG_SUFFIX, 1)[0]
        if orig_stem not in aug_image_map:
            aug_image_map[orig_stem] = f

sample_origs = random.sample(orig_images, min(3, len(orig_images)))
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('Augmentation Verification — Original vs Augmented + Bounding Boxes', fontsize=13, fontweight='bold')

for row, orig_path in enumerate(sample_origs):
    # Original
    img_orig = cv2.imread(str(orig_path))
    lbl_orig = train_lbl_dir / (orig_path.stem + '.txt')
    if img_orig is not None:
        img_orig_drawn = draw_yolo_boxes(img_orig, lbl_orig, RAW_CLASS_NAMES)
        axes[row][0].imshow(cv2.cvtColor(img_orig_drawn, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title(f'Original: {orig_path.name[:30]}', fontsize=8)
        axes[row][0].axis('off')
    
    # Augmented
    aug_path = aug_image_map.get(orig_path.stem)
    if aug_path:
        img_aug = cv2.imread(str(aug_path))
        lbl_aug = train_lbl_dir / (aug_path.stem + '.txt')
        if img_aug is not None:
            img_aug_drawn = draw_yolo_boxes(img_aug, lbl_aug, RAW_CLASS_NAMES)
            axes[row][1].imshow(cv2.cvtColor(img_aug_drawn, cv2.COLOR_BGR2RGB))
            axes[row][1].set_title(f'Augmented: {aug_path.name[:30]}', fontsize=8)
            axes[row][1].axis('off')
    else:
        axes[row][1].text(0.5, 0.5, 'No augmented found', ha='center', va='center')
        axes[row][1].axis('off')

plt.tight_layout()
plt.savefig(str(REPORTS_DIR / 'augmentation_verification.png'), dpi=80, bbox_inches='tight')
plt.show()
print('Saved to reports/augmentation_verification.png')

---
## Section 14 — Final YOLO Dataset Validation

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from validate_yolo_dataset import validate_dataset

print('Running final dataset validation...')
VALIDATION_PASSED = validate_dataset(FINAL_DATASET, verbose=False)

---
## Section 15 — Generate data.yaml

In [ ]:
# Build final data.yaml with relative paths
final_yaml = {
    'path': str(FINAL_DATASET),  # absolute path for portability
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NUM_CLASSES,
    'names': RAW_CLASS_NAMES,
}

# Remove test if no test split was copied
test_img_dir = FINAL_DATASET / 'images' / 'test'
if not test_img_dir.exists() or not any(test_img_dir.iterdir()):
    final_yaml.pop('test', None)
    print('[INFO] No test split — omitted from data.yaml')

yaml_out = FINAL_DATASET / 'data.yaml'
with open(yaml_out, 'w', encoding='utf-8') as f:
    yaml.dump(final_yaml, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print('Final data.yaml:')
print('='*50)
print(yaml_out.read_text())
print('='*50)
print(f'Saved to: {yaml_out}')

---
## Section 16 — YOLOv8 Smoke Test

In [ ]:
from ultralytics import YOLO
import torch

SMOKE_TEST_EPOCHS = 1
SMOKE_TEST_IMG_SIZE = 640
SMOKE_TEST_BATCH = 4
SMOKE_MODEL = 'yolov8n.pt'

print('='*60)
print('YOLOv8 SMOKE TEST')
print('='*60)
print(f'Model    : {SMOKE_MODEL}')
print(f'Epochs   : {SMOKE_TEST_EPOCHS}')
print(f'ImgSize  : {SMOKE_TEST_IMG_SIZE}')
print(f'Batch    : {SMOKE_TEST_BATCH}')
print(f'data.yaml: {yaml_out}')
print(f'Device   : {"GPU" if torch.cuda.is_available() else "CPU"}')
print('='*60)

SMOKE_TEST_PASSED = False
smoke_run_dir = PROJECT_ROOT / 'runs' / 'smoke_test'

try:
    model = YOLO(SMOKE_MODEL)
    results = model.train(
        data=str(yaml_out),
        epochs=SMOKE_TEST_EPOCHS,
        imgsz=SMOKE_TEST_IMG_SIZE,
        batch=SMOKE_TEST_BATCH,
        project=str(PROJECT_ROOT / 'runs'),
        name='smoke_test',
        exist_ok=True,
        verbose=False,
        plots=False,
    )
    SMOKE_TEST_PASSED = True
    print('\n✅ SMOKE TEST PASSED — dataset loads and training works!')
except Exception as e:
    SMOKE_TEST_PASSED = False
    print(f'\n❌ SMOKE TEST FAILED: {e}')
    raise

print(f'Smoke test result: {"PASS" if SMOKE_TEST_PASSED else "FAIL"}')

---
## Section 17 — Final Dataset Export

In [ ]:
# ── Count final dataset ───────────────────────────────────────────────────────
final_counts = {}
for split in ['train', 'val', 'test']:
    img_d = FINAL_DATASET / 'images' / split
    lbl_d = FINAL_DATASET / 'labels' / split
    if img_d.exists():
        n_img = sum(1 for f in img_d.iterdir() if f.suffix.lower() in SUPPORTED_EXTS)
        n_lbl = sum(1 for f in lbl_d.iterdir() if f.suffix == '.txt') if lbl_d.exists() else 0
        final_counts[split] = {'images': n_img, 'labels': n_lbl}

total_final_images = sum(v['images'] for v in final_counts.values())
total_final_labels = sum(v['labels'] for v in final_counts.values())

print('Final Dataset Summary:')
print(f'  Total images     : {total_final_images}')
print(f'  Total labels     : {total_final_labels}')
for split, c in final_counts.items():
    print(f'  {split:8s}        : {c["images"]:4d} images | {c["labels"]:4d} labels')

# ── Calculate total size ──────────────────────────────────────────────────────
total_bytes = sum(f.stat().st_size for f in FINAL_DATASET.rglob('*') if f.is_file())
total_mb = total_bytes / (1024**2)
print(f'  Total size       : {total_mb:.1f} MB')

In [ ]:
# ── Create ZIP ────────────────────────────────────────────────────────────────
ZIP_NAME = 'textile_defect_yolov8_final'
ZIP_PATH = PROJECT_ROOT / f'{ZIP_NAME}.zip'

# Remove existing ZIP if present
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print(f'Creating ZIP: {ZIP_PATH}')
shutil.make_archive(
    str(PROJECT_ROOT / ZIP_NAME),
    'zip',
    root_dir=str(FINAL_DATASET.parent),
    base_dir=FINAL_DATASET.name
)

zip_size_mb = ZIP_PATH.stat().st_size / (1024**2)
print(f'✅ ZIP created: {ZIP_PATH}')
print(f'   Size: {zip_size_mb:.1f} MB')

In [ ]:
# ── Google Drive / OneDrive check ─────────────────────────────────────────────
import getpass

GOOGLE_DRIVE_PATHS = [
    Path(f'C:/Users/{getpass.getuser()}/Google Drive'),
    Path(f'C:/Users/{getpass.getuser()}/My Drive'),
    Path('G:/My Drive'),
    Path('G:/Google Drive'),
]

GOOGLE_DRIVE_ROOT = None
for p in GOOGLE_DRIVE_PATHS:
    if p.exists():
        GOOGLE_DRIVE_ROOT = p
        print(f'✅ Google Drive found: {p}')
        break

DRIVE_UPLOAD_SUCCESS = False
DRIVE_LINK = None

if GOOGLE_DRIVE_ROOT:
    drive_dest = GOOGLE_DRIVE_ROOT / 'Hangzhou_Textile_POC'
    drive_dest.mkdir(exist_ok=True)
    dest_zip = drive_dest / ZIP_PATH.name
    shutil.copy2(ZIP_PATH, dest_zip)
    if dest_zip.exists() and dest_zip.stat().st_size == ZIP_PATH.stat().st_size:
        DRIVE_UPLOAD_SUCCESS = True
        DRIVE_LINK = str(dest_zip)
        print(f'✅ ZIP copied to Google Drive: {dest_zip}')
        print(f'   Size match: {dest_zip.stat().st_size / 1024**2:.1f} MB')
    else:
        print('❌ Size mismatch after copy!')
else:
    print('⚠️  Google Drive desktop sync folder NOT found.')
    print('    Manual upload required.')
    print(f'    Upload this file to ehtisham.malik5618@gmail.com Drive:')
    print(f'    → {ZIP_PATH}')
    print(f'    → Create folder: Hangzhou_Textile_POC')
    print(f'    → Place file inside: textile_defect_yolov8_final.zip')
    DRIVE_LINK = 'MANUAL_UPLOAD_REQUIRED'

print(f'\nDrive upload status: {"SUCCESS" if DRIVE_UPLOAD_SUCCESS else "MANUAL_REQUIRED"}')

In [ ]:
# ── Final completion summary ──────────────────────────────────────────────────
print('\n' + '='*65)
print('PIPELINE COMPLETION SUMMARY')
print('='*65)
print(f'Dataset            : CHENAB Textile / FabricDefectNTU')
print(f'Kaggle ID          : muhammadharisabid/fabricdefectntu')
print(f'Download date      : {DOWNLOAD_DATE}')
print(f'Classes            : {NUM_CLASSES} → {RAW_CLASS_NAMES}')
print()
print('Original dataset:')
print(f'  Total images     : {stats["total_images"]}')
print(f'  Total annotations: {stats["total_annotations"]}')
print()
print('Final dataset:')
for split, c in final_counts.items():
    print(f'  {split:8s}        : {c["images"]:4d} images | {c["labels"]:4d} labels')
print(f'  Total images     : {total_final_images}')
print(f'  Dataset size     : {total_mb:.1f} MB')
print(f'  ZIP size         : {zip_size_mb:.1f} MB')
print()
print(f'Validation         : {"PASS" if VALIDATION_PASSED else "FAIL"}')
print(f'YOLOv8 smoke test  : {"PASS" if SMOKE_TEST_PASSED else "FAIL"}')
print(f'ZIP created        : {ZIP_PATH}')
print(f'Drive upload       : {"SUCCESS" if DRIVE_UPLOAD_SUCCESS else "MANUAL_REQUIRED"}')
print('='*65)